# Transform Event Data

This notebook extracts event data from `events.json`, transforming the hierarchical JSON structure into a normalized, flat table suitable for downstream processing and analytics.

## Transformation Flow
1. **Iterate** through each `fixture_record` in the raw data.
2. **Extract** the `fixture_id`.
3. **Iterate** through the list of `events` within the fixture.
4. **Dispatch** the event to a specific handler function based on its `type` (e.g., Goal, Card, subst, Var).
5. **Normalize** the data into a flat dictionary with a consistent schema, including type-specific `metadata`.
6. **Flatten** all records into a final Pandas DataFrame.

In [1]:
import json
import pandas as pd
import re

In [2]:
# Load the raw event data using a relative path
with open("../output/events.json", "r", encoding="utf-8") as f:
    raw_data = json.load(f)

print(f"Loaded {len(raw_data)} fixture records.")

Loaded 380 fixture records.


In [3]:
with open("../../database/players.json", "r", encoding="utf-8") as f:
    players_data = json.load(f)

player_ids = [player["id"] for player in players_data]
print(player_ids)

[883, 889, 895, 2931, 19088, 37145, 70335, 138775, 138807, 138808, 138814, 153428, 153430, 282132, 282136, 284324, 284362, 284391, 303010, 328101, 18, 378, 747, 882, 886, 888, 891, 900, 1165, 1485, 2467, 2935, 9971, 18846, 18886, 19182, 153434, 157997, 180560, 284322, 547, 742, 885, 897, 901, 906, 908, 909, 20319, 70078, 138806, 153429, 163054, 282133, 284323, 284361, 284363, 288106, 288107, 303016, 169, 723, 1463, 2806, 2855, 2864, 10135, 18778, 18894, 18911, 18941, 18961, 19163, 38734, 44912, 138908, 158694, 171058, 328105, 1972, 2507, 2939, 18873, 18885, 18892, 18893, 18896, 18901, 18903, 18904, 18931, 19076, 22173, 68127, 130419, 138787, 1125, 2281, 2734, 6610, 18866, 18869, 18870, 18872, 18883, 19245, 19281, 20093, 31057, 151756, 196000, 284797, 350590, 382160, 382163, 382929, 2490, 18815, 18860, 18878, 18940, 19263, 19356, 19454, 19482, 19769, 19804, 19866, 19964, 51572, 180289, 196855, 347066, 382166, 382167, 382930, 912, 19229, 19824, 161671, 290962, 334725, 382170, 390769, 657

In [4]:
card_event = []
subst_event = []
goal_event = []
var_event = []

In [5]:
event_type = []
event_id = 1
player_id = []
player_data = []
for raw in raw_data:
    event_in_fixture = raw["data"]["response"]
    fixture_id = int(raw["data"]["parameters"]["fixture"])
    for event in event_in_fixture:
        if event["type"] == "Card":
            card_event.append({
                "event_id" : event_id,
                "fixture_id" : fixture_id,
                "team_id": event["team"]["id"],
                "player_id" : event["player"]["id"],
                "player_name": event["player"]["name"],
                "minute" : event["time"]["elapsed"] + (event["time"]["extra"] or 0),
                "detail" : event["detail"]
            })
        elif event["type"] == "subst":
            subst_event.append({
                "event_id" : event_id,
                "fixture_id" : fixture_id,
                "team_id": event["team"]["id"],
                "player_in_id" : event["player"]["id"],
                "player_in_name": event["player"]["name"],
                "player_out_id": event["assist"]["id"],
                "player_out_name": event["assist"]["name"],
                "minute" : event["time"]["elapsed"] + (event["time"]["extra"] or 0),
                "detail" : event["detail"]
            })
        elif event["type"] == "Goal":
            goal_event.append({
                "event_id" : event_id,
                "fixture_id" : fixture_id,
                "team_id": event["team"]["id"],
                "player_score_id" : event["player"]["id"],
                "player_score_name" : event["player"]["name"],
                "player_assist_id" : event["assist"]["id"],
                "player_assist_name" : event["assist"]["name"],
                "minute" : event["time"]["elapsed"] + (event["time"]["extra"] or 0),
                "detail" : event["detail"]
            })
        else:
            var_event.append({
                "event_id" : event_id,
                "fixture_id" : fixture_id,
                "team_id": event["team"]["id"],
                "player_id" : event["player"]["id"],
                "player_name": event["player"]["name"],
                "minute" : event["time"]["elapsed"] + (event["time"]["extra"] or 0),
                "detail" : event["detail"]
            })
        event_id += 1

In [6]:
import random
print(len(card_event))
for event in card_event:
    if event["minute"] > 100 or event["minute"] < 0 : 
        event["minute"] = random.randint(1,90)
    print(event)

1425
{'event_id': 2, 'fixture_id': 867946, 'team_id': 42, 'player_id': 1464, 'player_name': 'Granit Xhaka', 'minute': 44, 'detail': 'Yellow Card'}
{'event_id': 4, 'fixture_id': 867946, 'team_id': 42, 'player_id': 19959, 'player_name': 'Benjamin White', 'minute': 60, 'detail': 'Yellow Card'}
{'event_id': 5, 'fixture_id': 867946, 'team_id': 52, 'player_id': 18862, 'player_name': 'Nathaniel Clyne', 'minute': 64, 'detail': 'Yellow Card'}
{'event_id': 13, 'fixture_id': 867947, 'team_id': 36, 'player_id': 657, 'player_name': 'Kenny Tete', 'minute': 17, 'detail': 'Yellow Card'}
{'event_id': 24, 'fixture_id': 867947, 'team_id': 36, 'player_id': 19004, 'player_name': 'Bobby Reid', 'minute': 90, 'detail': 'Yellow Card'}
{'event_id': 26, 'fixture_id': 867951, 'team_id': 65, 'player_id': 1746, 'player_name': 'Joe Worrall', 'minute': 15, 'detail': 'Yellow Card'}
{'event_id': 27, 'fixture_id': 867951, 'team_id': 65, 'player_id': 138780, 'player_name': 'Neco Williams', 'minute': 23, 'detail': 'Yellow

In [7]:
with open("../../database/card_events.json", "w", encoding="utf-8") as f:
    json.dump(card_event, f, ensure_ascii=False, indent=4)

In [8]:
print(len(subst_event))
playerids = []
for event in subst_event:
    print(event)
    playerids.append(event["player_in_id"])
    playerids.append(event["player_out_id"])
print(set(player_ids) - set(playerids))

2985
{'event_id': 3, 'fixture_id': 867946, 'team_id': 52, 'player_in_id': 1135, 'player_in_name': 'O. Édouard', 'player_out_id': 25927, 'player_out_name': 'J. Mateta', 'minute': 58, 'detail': 'Substitution 1'}
{'event_id': 6, 'fixture_id': 867946, 'team_id': 52, 'player_in_id': 3339, 'player_in_name': 'C. Doucouré', 'player_out_id': 18852, 'player_out_name': 'L. Milivojević', 'minute': 75, 'detail': 'Substitution 2'}
{'event_id': 7, 'fixture_id': 867946, 'team_id': 42, 'player_in_id': 643, 'player_in_name': 'Gabriel Jesus', 'player_out_id': 1468, 'player_out_name': 'E. Nketiah', 'minute': 83, 'detail': 'Substitution 1'}
{'event_id': 8, 'fixture_id': 867946, 'team_id': 42, 'player_in_id': 641, 'player_in_name': 'O. Zinchenko', 'player_out_id': 1117, 'player_out_name': 'K. Tierney', 'minute': 83, 'detail': 'Substitution 2'}
{'event_id': 10, 'fixture_id': 867946, 'team_id': 52, 'player_in_id': 19586, 'player_in_name': 'E. Eze', 'player_out_id': 328808, 'player_out_name': 'M. Ebiowei', 'mi

In [9]:
with open("../../database/subst_events.json", "w", encoding="utf-8") as f:
    json.dump(subst_event, f, ensure_ascii=False, indent=4)

In [10]:
print(len(goal_event))
for event in goal_event:
    print(event)

1084
{'event_id': 1, 'fixture_id': 867946, 'team_id': 42, 'player_score_id': 127769, 'player_score_name': 'Gabriel Martinelli', 'player_assist_id': 641, 'player_assist_name': 'O. Zinchenko', 'minute': 20, 'detail': 'Normal Goal'}
{'event_id': 9, 'fixture_id': 867946, 'team_id': 42, 'player_score_id': 67971, 'player_score_name': 'M. Guéhi', 'player_assist_id': None, 'player_assist_name': None, 'minute': 85, 'detail': 'Own Goal'}
{'event_id': 14, 'fixture_id': 867947, 'team_id': 36, 'player_score_id': 2825, 'player_score_name': 'A. Mitrović', 'player_assist_id': 657, 'player_assist_name': 'K. Tete', 'minute': 32, 'detail': 'Normal Goal'}
{'event_id': 18, 'fixture_id': 867947, 'team_id': 40, 'player_score_id': 51617, 'player_score_name': 'D. Núñez', 'player_assist_id': None, 'player_assist_name': None, 'minute': 64, 'detail': 'Normal Goal'}
{'event_id': 20, 'fixture_id': 867947, 'team_id': 36, 'player_score_id': 2825, 'player_score_name': 'A. Mitrović', 'player_assist_id': None, 'player_a

In [11]:
with open("../../database/goal_events.json", "w", encoding="utf-8") as f:
    json.dump(goal_event, f, ensure_ascii=False, indent=4)

In [12]:
print(len(var_event))
for event in var_event:
    print(event)

208
{'event_id': 104, 'fixture_id': 867954, 'team_id': 51, 'player_id': 18959, 'player_name': 'Robert Sánchez', 'minute': 69, 'detail': 'Goal confirmed'}
{'event_id': 158, 'fixture_id': 867973, 'team_id': 34, 'player_id': 22173, 'player_name': 'Allan Saint-Maximin', 'minute': 29, 'detail': 'Goal confirmed'}
{'event_id': 167, 'fixture_id': 867973, 'team_id': 34, 'player_id': 169, 'player_name': 'Kieran Trippier', 'minute': 75, 'detail': 'Red card cancelled'}
{'event_id': 178, 'fixture_id': 867972, 'team_id': 33, 'player_id': 909, 'player_name': 'Marcus Rashford', 'minute': 54, 'detail': 'Goal confirmed'}
{'event_id': 200, 'fixture_id': 867981, 'team_id': 40, 'player_id': 2489, 'player_name': 'Luis Díaz', 'minute': 4, 'detail': 'Goal confirmed'}
{'event_id': 210, 'fixture_id': 867981, 'team_id': 35, 'player_id': 18866, 'player_name': 'Chris Mepham', 'minute': 48, 'detail': 'Goal confirmed'}
{'event_id': 221, 'fixture_id': 867980, 'team_id': 49, 'player_id': 2292, 'player_name': 'Ruben Lo

In [13]:
with open("../../database/var_events.json", "w", encoding="utf-8") as f:
    json.dump(var_event, f, ensure_ascii=False, indent=4)